# Karlo Roel Montenegro & Meagelleine Rose Sionosa
## BSCS 3A-AI
## 02/12/26

### Activity 4 - Naive Bayes

# 1. Performing it manually. In manually developing a Naïve Bayes model, create methods that will do the following:
- Generate a Bag of Words (for word frequency)
- Calculate the prior for the class HAM and SPAM
- Calculate the likelihood of the tokens in the vocabulary with respect to the class.


- Determine the class of the following test sentence:
     - Limited offer, click here!
     - Meeting at 2 PM with the manager.


### Dataset
| doc | class |
|-----|-------|
| Free money now!!! | SPAM |
| Hi mom, how are you? | HAM |
| Lowest price for your meds | SPAM |
| Are we still on for dinner? | HAM |
| Win a free iPhone today | SPAM |
| Let's catch up tomorrow at the office | HAM |
| Meeting at 3 PM tomorrow | HAM |
| Get 50% off, limited time! | SPAM |
| Team meeting in the office | HAM |
| Click here for prizes! | SPAM |
| Can you send the report? | HAM |

In [29]:
# import libraries

import re
from collections import defaultdict
import math

In [30]:
# ── Dataset ──────────────────────────────────────────────────────────────────
documents = [
    ("Free money now!!!",                       "SPAM"),
    ("Hi mom, how are you?",                    "HAM"),
    ("Lowest price for your meds",              "SPAM"),
    ("Are we still on for dinner?",             "HAM"),
    ("Win a free iPhone today",                 "SPAM"),
    ("Let's catch up tomorrow at the office",   "HAM"),
    ("Meeting at 3 PM tomorrow",                "HAM"),
    ("Get 50% off, limited time!",              "SPAM"),
    ("Team meeting in the office",              "HAM"),
    ("Click here for prizes!",                  "SPAM"),
    ("Can you send the report?",                "HAM"),
]

print(f"Total documents : {len(documents)}")
print(f"SPAM documents  : {sum(1 for _, c in documents if c == 'SPAM')}")
print(f"HAM  documents  : {sum(1 for _, c in documents if c == 'HAM')}")

Total documents : 11
SPAM documents  : 5
HAM  documents  : 6


In [31]:
# tokenize

def tokenize(text: str) -> list[str]:
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)   # ← space, not ""
    return text.split()


In [ ]:
# bag of words

from collections import defaultdict

def build_bag_of_words(documents: list[tuple]) -> dict:
    """
    Build per-class word-frequency dictionaries and a global vocabulary.

    Returns
    -------
    bow : dict  {class_label: {word: count, ...}}
    vocab : set  all unique words across every document
    """
    bow   = defaultdict(lambda: defaultdict(int))
    vocab = set()

    for text, label in documents:
        for token in tokenize(text):
            bow[label][token] += 1
            vocab.add(token)

    return dict(bow), vocab


bow, vocab = build_bag_of_words(documents)
vocab = sorted(vocab)   # sorted for consistent display

print(f"Vocabulary size : {len(vocab)}")
print(f"\nBag of Words:\n{vocab}")

print("\n── SPAM word frequencies ──")
for w, c in sorted(bow['SPAM'].items()):
    print(f"  {w:<15} {c}")

print("\n── HAM word frequencies ──")
for w, c in sorted(bow['HAM'].items()):
    print(f"  {w:<15} {c}")

Vocabulary size : 45

Vocabulary:
['3', '50', 'a', 'are', 'at', 'can', 'catch', 'click', 'dinner', 'for', 'free', 'get', 'here', 'hi', 'how', 'in', 'iphone', 'let', 'limited', 'lowest', 'meds', 'meeting', 'mom', 'money', 'now', 'off', 'office', 'on', 'pm', 'price', 'prizes', 'report', 's', 'send', 'still', 'team', 'the', 'time', 'today', 'tomorrow', 'up', 'we', 'win', 'you', 'your']

── SPAM word frequencies ──
  50              1
  a               1
  click           1
  for             2
  free            2
  get             1
  here            1
  iphone          1
  limited         1
  lowest          1
  meds            1
  money           1
  now             1
  off             1
  price           1
  prizes          1
  time            1
  today           1
  win             1
  your            1

── HAM word frequencies ──
  3               1
  are             2
  at              2
  can             1
  catch           1
  dinner          1
  for             1
  hi             

In [33]:
# prior calculation

def calculate_priors(documents: list[tuple]) -> dict:
    """
    P(class) = (number of documents in class) / (total documents)

    Returns
    -------
    priors : dict  {class_label: probability}
    """
    total  = len(documents)
    counts = defaultdict(int)

    for _, label in documents:
        counts[label] += 1

    priors = {label: count / total for label, count in counts.items()}
    return priors


priors = calculate_priors(documents)

print("Prior Probabilities")
print("=" * 35)
n_total = len(documents)
for label, prob in priors.items():
    count = sum(1 for _, c in documents if c == label)
    print(f"P({label:<4}) = {count}/{n_total} = {prob:.4f}")

Prior Probabilities
P(SPAM) = 5/11 = 0.4545
P(HAM ) = 6/11 = 0.5455


In [34]:
# calculating likelihoods with Laplace smoothing

def calculate_likelihoods(bow: dict, vocab: list) -> dict:
    """
    Compute smoothed word likelihoods for every class.

    P(word | class) = (count(word, class) + 1)
                      / (total_words_in_class + |vocab|)

    Returns
    -------
    likelihoods : dict  {class_label: {word: probability}}
    """
    likelihoods = {}
    V = len(vocab)                               # vocabulary size

    for label, word_counts in bow.items():
        total_words = sum(word_counts.values())  # total word tokens in this class
        denominator = total_words + V

        likelihoods[label] = {
            word: (word_counts.get(word, 0) + 1) / denominator
            for word in vocab
        }

    return likelihoods


likelihoods = calculate_likelihoods(bow, vocab)

# ── Pretty-print a comparison table ─────────────────────────────────────────
header = f"{'Word':<18} {'P(w|SPAM)':>12} {'P(w|HAM)':>12}  {'SPAM count':>10} {'HAM count':>9}"
print("Likelihood Table  (Laplace smoothed)")
print("=" * len(header))
print(header)
print("-" * len(header))

for word in sorted(vocab):
    p_spam = likelihoods['SPAM'][word]
    p_ham  = likelihoods['HAM'][word]
    c_spam = bow['SPAM'].get(word, 0)
    c_ham  = bow['HAM'].get(word, 0)
    print(f"{word:<18} {p_spam:>12.6f} {p_ham:>12.6f}  {c_spam:>10} {c_ham:>9}")

Likelihood Table  (Laplace smoothed)
Word                  P(w|SPAM)     P(w|HAM)  SPAM count HAM count
------------------------------------------------------------------
3                      0.014925     0.025316           0         1
50                     0.029851     0.012658           1         0
a                      0.029851     0.012658           1         0
are                    0.014925     0.037975           0         2
at                     0.014925     0.037975           0         2
can                    0.014925     0.025316           0         1
catch                  0.014925     0.025316           0         1
click                  0.029851     0.012658           1         0
dinner                 0.014925     0.025316           0         1
for                    0.044776     0.025316           2         1
free                   0.044776     0.012658           2         0
get                    0.029851     0.012658           1         0
here                   0.

In [35]:
# calculate classification scores for a new document

def classify(text: str,
             priors:      dict,
             likelihoods: dict,
             bow:         dict,
             vocab:       list) -> tuple:
    """
    Predict the class of `text` using Naïve Bayes (regular probability scale).

    Returns
    -------
    predicted_class : str
    scores          : dict  {class_label: probability_score}
    token_detail    : list  of [token, P(token|HAM), P(token|SPAM)]
    """
    tokens       = tokenize(text)
    scores       = {}
    V            = len(vocab)
    token_detail = []

    for label in priors:
        total_words_in_class = sum(bow[label].values())
        denominator          = total_words_in_class + V

        score = priors[label]                    # prior probability

        for token in tokens:
            # smoothed P(token | class)
            count = bow[label].get(token, 0)
            p_token_given_class = (count + 1) / denominator
            score *= p_token_given_class         # multiply probabilities

        scores[label] = score

    # Build token detail (for display)
    for token in tokens:
        row = [token]
        for label in sorted(priors):             # sorted → HAM, SPAM
            total_words_in_class = sum(bow[label].values())
            denominator          = total_words_in_class + V
            count = bow[label].get(token, 0)
            p = (count + 1) / denominator
            row.append(f"{p:.6f}")
        token_detail.append(row)

    predicted = max(scores, key=scores.get)
    return predicted, scores, token_detail


def print_classification(sentence: str) -> None:
    """Pretty-print the full classification breakdown for one sentence."""
    predicted, scores, token_detail = classify(sentence, priors, likelihoods, bow, vocab)

    tokens = tokenize(sentence)
    print(f" Tokens        : {tokens}")

    # Per-token probabilities
    print(f"\n {'Token':<15} {'P(token|HAM)':>15} {'P(token|SPAM)':>15}")
    print(f" {'-'*15} {'-'*15} {'-'*15}")
    for row in token_detail:
        # row = [token, HAM_prob, SPAM_prob]  (sorted labels: HAM before SPAM)
        print(f" {row[0]:<15} {row[1]:>15} {row[2]:>15}")

    # Scores
    print("\n Probability Scores")
    print(f" {'Class':<8} {'Prior':>12} {'Final Score':>14}")
    print(f" {'-'*8} {'-'*12} {'-'*14}")
    for label in sorted(scores):
        prior = priors[label]
        print(f" {label:<8} {prior:>12.4f} {scores[label]:>14.10f}")

    print(f"\n  Predicted Class : {predicted}")

print("classify and print_classification are ready.")

classify and print_classification are ready.


In [36]:
# classifyn i
sentence_i = "Limited offer, click here!"

print("═" * 60)
print(f" Test Sentence (i) : \"{sentence_i}\"")
print("═" * 60)
print_classification(sentence_i)

════════════════════════════════════════════════════════════
 Test Sentence (i) : "Limited offer, click here!"
════════════════════════════════════════════════════════════
 Tokens        : ['limited', 'offer', 'click', 'here']

 Token              P(token|HAM)   P(token|SPAM)
 --------------- --------------- ---------------
 limited                0.012658        0.029851
 offer                  0.012658        0.014925
 click                  0.012658        0.029851
 here                   0.012658        0.029851

 Probability Scores
 Class           Prior    Final Score
 -------- ------------ --------------
 HAM            0.5455   0.0000000140
 SPAM           0.4545   0.0000001805

  Predicted Class : SPAM


In [37]:
# classify ii
sentence_ii = "Meeting at 2 PM with the manager."

print("═" * 60)
print(f" Test Sentence (ii) : \"{sentence_ii}\"")
print("═" * 60)
print_classification(sentence_ii)

════════════════════════════════════════════════════════════
 Test Sentence (ii) : "Meeting at 2 PM with the manager."
════════════════════════════════════════════════════════════
 Tokens        : ['meeting', 'at', '2', 'pm', 'with', 'the', 'manager']

 Token              P(token|HAM)   P(token|SPAM)
 --------------- --------------- ---------------
 meeting                0.037975        0.014925
 at                     0.037975        0.014925
 2                      0.012658        0.014925
 pm                     0.025316        0.014925
 with                   0.012658        0.014925
 the                    0.050633        0.014925
 manager                0.012658        0.014925

 Probability Scores
 Class           Prior    Final Score
 -------- ------------ --------------
 HAM            0.5455   0.0000000000
 SPAM           0.4545   0.0000000000

  Predicted Class : HAM


# 2. Using Scikit-Learn. Use the scikit-learn package to train and test a Multinomial Naïve Bayes classifer.
- Determine the class of the following test sentence:
    - Limited offer, click here!
    - Meeting at 2 PM with the manager.


---
---
We use the following sklearn pipeline:

| Component | Role |
|---|---|
| `CountVectorizer` | Converts raw text → token count matrix (Bag of Words) |
| `MultinomialNB` | Trains & predicts using Multinomial Naïve Bayes |

The entire dataset is used for **training** (same 11 documents as Part 1), then the two test sentences are classified.

In [38]:
#Imports and dataset

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
import numpy as np

# ── Training data (same 11 documents as Part 1) ───────────────────────────────
train_texts = [
    "Free money now!!!",
    "Hi mom, how are you?",
    "Lowest price for your meds",
    "Are we still on for dinner?",
    "Win a free iPhone today",
    "Let's catch up tomorrow at the office",
    "Meeting at 3 PM tomorrow",
    "Get 50% off, limited time!",
    "Team meeting in the office",
    "Click here for prizes!",
    "Can you send the report?",
]

train_labels = [
    "SPAM", "HAM", "SPAM", "HAM", "SPAM",
    "HAM",  "HAM", "SPAM", "HAM",  "SPAM", "HAM",
]

print(f"Training samples : {len(train_texts)}")
print(f"  SPAM : {train_labels.count('SPAM')}")
print(f"  HAM  : {train_labels.count('HAM')}")

Training samples : 11
  SPAM : 5
  HAM  : 6


In [39]:
#Build pipeline and train model
# ── Pipeline: CountVectorizer → MultinomialNB ─────────────────────────────────
# CountVectorizer parameters mirror Part 1:
#   - lowercase=True        : case-fold all tokens
#   - token_pattern         : keep only alphabetic/numeric tokens
#   - min_df=1              : include every token (no frequency cut-off)
# MultinomialNB:
#   - alpha=1.0 (default)   : Laplace (add-1) smoothing — identical to Part 1

pipeline = Pipeline([
    ("vectorizer", CountVectorizer(
        lowercase=True,
        token_pattern=r"(?u)\b[a-z0-9]+\b",  # strip punctuation, match Part 1
        min_df=1,
    )),
    ("classifier", MultinomialNB(alpha=1.0)),  # alpha=1.0 → Laplace smoothing
])

pipeline.fit(train_texts, train_labels)
print("Model trained successfully!")

# ── Inspect the learned vocabulary ───────────────────────────────────────────
vectorizer = pipeline.named_steps["vectorizer"]
sk_vocab    = vectorizer.get_feature_names_out()
print(f"\nVocabulary size : {len(sk_vocab)}")
print(f"Vocabulary      : {sorted(sk_vocab.tolist())}")

Model trained successfully!

Vocabulary size : 45
Vocabulary      : ['3', '50', 'a', 'are', 'at', 'can', 'catch', 'click', 'dinner', 'for', 'free', 'get', 'here', 'hi', 'how', 'in', 'iphone', 'let', 'limited', 'lowest', 'meds', 'meeting', 'mom', 'money', 'now', 'off', 'office', 'on', 'pm', 'price', 'prizes', 'report', 's', 'send', 'still', 'team', 'the', 'time', 'today', 'tomorrow', 'up', 'we', 'win', 'you', 'your']


In [40]:
#Learned priors & Feature Probabilities

nb_model  = pipeline.named_steps["classifier"]
classes   = nb_model.classes_          # ['HAM', 'SPAM']

# ── Prior probabilities ───────────────────────────────────────────────────────
print("Prior Probabilities (learned by MultinomialNB)")
print("=" * 48)
for cls, log_prior in zip(classes, nb_model.class_log_prior_):
    prior = np.exp(log_prior)  # Convert from log to regular probability
    print(f"  P({cls:<4}) = {prior:.4f}")

# ── Feature likelihoods ───────────────────────────────────────────────────────
print("\nFeature Likelihoods  [P(word | class)]")
print("(showing words relevant to the test sentences)")
print()

highlight_words = ["limited", "offer", "click", "here",
                   "meeting", "at", "pm", "with", "the", "manager", "2"]

header = f"  {'Word':<12} {'P(w|HAM)':>15} {'P(w|SPAM)':>15}"
print(header)
print("  " + "-" * (len(header) - 2))

vocab_list = sk_vocab.tolist()
for word in highlight_words:
    if word in vocab_list:
        idx = vocab_list.index(word)
        # Convert from log probabilities to regular probabilities
        p_ham  = np.exp(nb_model.feature_log_prob_[0][idx])
        p_spam = np.exp(nb_model.feature_log_prob_[1][idx])
        print(f"  {word:<12} {p_ham:>15.6f} {p_spam:>15.6f}")
    else:
        print(f"  {word:<12}  (not in training vocabulary — smoothed to alpha / denom)")

Prior Probabilities (learned by MultinomialNB)
  P(HAM ) = 0.5455
  P(SPAM) = 0.4545

Feature Likelihoods  [P(word | class)]
(showing words relevant to the test sentences)

  Word                P(w|HAM)       P(w|SPAM)
  --------------------------------------------
  limited             0.012658        0.029851
  offer         (not in training vocabulary — smoothed to alpha / denom)
  click               0.012658        0.029851
  here                0.012658        0.029851
  meeting             0.037975        0.014925
  at                  0.037975        0.014925
  pm                  0.025316        0.014925
  with          (not in training vocabulary — smoothed to alpha / denom)
  the                 0.050633        0.014925
  manager       (not in training vocabulary — smoothed to alpha / denom)
  2             (not in training vocabulary — smoothed to alpha / denom)


In [41]:
#Classification (using regular probabilities, not log)

def sk_classify(sentence: str) -> None:
    """Classify a sentence using regular probability multiplication (like manual method)."""
    
    # Get tokens the vectorizer recognizes
    x_vec  = vectorizer.transform([sentence])
    tokens = [sk_vocab[i] for i in x_vec.nonzero()[1]]
    
    print(f" Tokens recognised by vectorizer : {tokens}")
    
    # Get class priors (regular probabilities)
    prior_ham  = np.exp(nb_model.class_log_prior_[0])  # HAM is first
    prior_spam = np.exp(nb_model.class_log_prior_[1])  # SPAM is second
    
    # Initialize scores with priors
    score_ham  = prior_ham
    score_spam = prior_spam
    
    # Per-token likelihoods
    print(f"\n {'Token':<12} {'P(token|HAM)':>15} {'P(token|SPAM)':>15}")
    print(f" {'-'*12} {'-'*15} {'-'*15}")
    
    vocab_list = sk_vocab.tolist()
    for token in tokens:
        if token in vocab_list:
            idx = vocab_list.index(token)
            # Convert from log probabilities to regular probabilities
            p_ham  = np.exp(nb_model.feature_log_prob_[0][idx])
            p_spam = np.exp(nb_model.feature_log_prob_[1][idx])
            print(f" {token:<12} {p_ham:>15.6f} {p_spam:>15.6f}")
            
            # Multiply probabilities (like manual method)
            score_ham  *= p_ham
            score_spam *= p_spam
    
    # Display results
    print("\n Probability Scores")
    print(f" {'Class':<8} {'Prior':>12} {'Final Score':>14}")
    print(f" {'-'*8} {'-'*12} {'-'*14}")
    print(f" {'HAM':<8} {prior_ham:>12.4f} {score_ham:>14.10f}")
    print(f" {'SPAM':<8} {prior_spam:>12.4f} {score_spam:>14.10f}")
    
    # Predict based on higher score
    prediction = "HAM" if score_ham > score_spam else "SPAM"
    print(f"\n  Predicted Class : {prediction}")


# ── Classify i ────────────────────────────────────────────────────────────────
sentence_i = "Limited offer, click here!"

print("═" * 60)
print(f" Test Sentence (i) : \"{sentence_i}\"")
print("═" * 60)
sk_classify(sentence_i)

════════════════════════════════════════════════════════════
 Test Sentence (i) : "Limited offer, click here!"
════════════════════════════════════════════════════════════
 Tokens recognised by vectorizer : ['click', 'here', 'limited']

 Token           P(token|HAM)   P(token|SPAM)
 ------------ --------------- ---------------
 click               0.012658        0.029851
 here                0.012658        0.029851
 limited             0.012658        0.029851

 Probability Scores
 Class           Prior    Final Score
 -------- ------------ --------------
 HAM            0.5455   0.0000011063
 SPAM           0.4545   0.0000120905

  Predicted Class : SPAM


In [42]:

# ── Classify ii ───────────────────────────────────────────────────────────────
sentence_ii = "Meeting at 2 PM with the manager."

print("═" * 60)
print(f" Test Sentence (ii) : \"{sentence_ii}\"")
print("═" * 60)
sk_classify(sentence_ii)

════════════════════════════════════════════════════════════
 Test Sentence (ii) : "Meeting at 2 PM with the manager."
════════════════════════════════════════════════════════════
 Tokens recognised by vectorizer : ['at', 'meeting', 'pm', 'the']

 Token           P(token|HAM)   P(token|SPAM)
 ------------ --------------- ---------------
 at                  0.037975        0.014925
 meeting             0.037975        0.014925
 pm                  0.025316        0.014925
 the                 0.050633        0.014925

 Probability Scores
 Class           Prior    Final Score
 -------- ------------ --------------
 HAM            0.5455   0.0000010083
 SPAM           0.4545   0.0000000226

  Predicted Class : HAM
